# Paso 1: Definición del problema:


### Contexto del problema

El proceso de contratación de trabajadores extranjeros bajo solicitudes LCA/H-1B en Estados Unidos requiere que las empresas declaren un salario ofrecido para cada posición laboral. Este salario debe estar alineado con factores como la ocupación, la industria, la ubicación del trabajo, el tipo de visa, el nivel salarial prevaleciente y las características regulatorias del empleador.

Para una empresa que desea contratar talento extranjero, definir un salario competitivo y coherente con el mercado es una decisión importante. Un salario mal estimado puede generar problemas como:

- Ofertas poco competitivas frente al mercado laboral.
- Errores en la planificación de costos de contratación.
- Diferencias importantes frente al salario prevaleciente requerido.
- Dificultad para comparar salarios entre ocupaciones, estados e industrias.
- Pérdida de eficiencia en procesos de recursos humanos y cumplimiento regulatorio.

Por esta razón, el proyecto se enfoca en construir un modelo de Machine Learning capaz de estimar el salario anual mínimo ofrecido a un trabajador extranjero en solicitudes LCA/H-1B certificadas.

-------------------------------------------------

# Paso 2: Obtencion y carga del conjunto de datos:
- El conjunto de datos se obtuvo desde la página de Kaggle, se encuentra almacenado en un formato ".csv".
"https://www.kaggle.com/datasets/zongaobian/h1b-lca-disclosure-data-2020-2024?select=Combined_LCA_Disclosure_Data_FY2020_to_FY2024.csv"



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### A. Carga del conjunto de datos

- !pip install optuna
- !pip install plotly

In [2]:
import pandas as pd
import numpy as np
import sqlite3
import re
import plotly.express as px
import xgboost as xgb
import optuna
import pickle
import gzip

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
from pathlib import Path

file_path = '/content/drive/MyDrive/Colab Notebooks/PROYECTO 4GEEKS/Combined_LCA_Disclosure_Data_FY2020_to_FY2024.csv'

## Paso 3: Almacenar la información

Para garantizar una mejor organización, validación y acceso estructurado a la información, el dataset final fue migrado desde un archivo plano CSV hacia una base de datos relacional utilizando SQLite desde Python.

Para esto se carga el archivo `Combined_LCA_Disclosure_Data_FY2020_to_FY2024.csv` desde Google Drive y se almacena en una tabla llamada `lca_disclosure_data`.

Posteriormente se realizan consultas SQL de validación usando `SELECT`, `INSERT` y `JOIN`.

In [3]:
# Crear carpeta para guardar la base de datos
Path("database").mkdir(exist_ok=True)

# Crear conexión SQLite
conn = sqlite3.connect("database/h1b_lca_project.db")

# Nombre de la tabla principal
table_name = "lca_disclosure_data"

In [4]:
# Tamaño de cada bloque de lectura dado que el archivo es muy grande
chunk_size = 100000

# Cargar CSV por partes hacia SQLite
for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, low_memory=False)):

    if i == 0:
        chunk.to_sql(
            name=table_name,
            con=conn,
            if_exists="replace",
            index=False
        )
    else:
        chunk.to_sql(
            name=table_name,
            con=conn,
            if_exists="append",
            index=False
        )

print(".csv migrado completamente a SQLite.")

.csv migrado completamente a SQLite.


In [5]:
query = """
PRAGMA table_info(lca_disclosure_data);
"""

estructura_tabla = pd.read_sql_query(query, conn)
estructura_tabla

,cid,name,type,notnull,dflt_value,pk
0,0,CASE_NUMBER,TEXT,0,None,0
1,1,CASE_STATUS,TEXT,0,None,0
2,2,RECEIVED_DATE,TEXT,0,None,0
3,3,DECISION_DATE,TEXT,0,None,0
4,4,ORIGINAL_CERT_DATE,TEXT,0,None,0
...,...,...,...,...,...,...
91,91,PREPARER_LAST_NAME,TEXT,0,None,0
92,92,PREPARER_FIRST_NAME,TEXT,0,None,0
93,93,PREPARER_MIDDLE_INITIAL,TEXT,0,None,0
94,94,PREPARER_BUSINESS_NAME,TEXT,0,None,0


In [6]:
query = """
SELECT COUNT(*) AS total_registros
FROM lca_disclosure_data;
"""

pd.read_sql_query(query, conn)

,total_registros
0,3564698


### SELECT

In [7]:
query = """
SELECT *
FROM lca_disclosure_data
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,CASE_NUMBER,CASE_STATUS,RECEIVED_DATE,DECISION_DATE,ORIGINAL_CERT_DATE,VISA_CLASS,JOB_TITLE,SOC_CODE,SOC_TITLE,FULL_TIME_POSITION,...,WILLFUL_VIOLATOR,SUPPORT_H1B,STATUTORY_BASIS,APPENDIX_A_ATTACHED,PUBLIC_DISCLOSURE,PREPARER_LAST_NAME,PREPARER_FIRST_NAME,PREPARER_MIDDLE_INITIAL,PREPARER_BUSINESS_NAME,PREPARER_EMAIL
0,I-200-19268-393467,Certified,2019-09-25,2019-10-01,None,H-1B,"APPLICATION ENGINEER, OMS [15-1199.02]",15-1199,"COMPUTER OCCUPATIONS, ALL OTHER",Y,...,N,None,None,None,Disclose Business,None,None,None,None,None
1,I-200-19268-638983,Certified,2019-09-25,2019-10-01,None,H-1B,BI DEVELOPER II,15-1132,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,...,N,Y,BOTH,None,Disclose Business,None,None,None,None,None
2,I-200-19268-177184,Certified,2019-09-25,2019-10-01,None,H-1B,QUALITY ENGINEER,17-2141,MECHANICAL ENGINEERS,Y,...,N,Y,BOTH,None,Disclose Business,None,None,None,None,None
3,I-200-19268-936403,Certified,2019-09-25,2019-10-01,None,H-1B,"SOFTWARE DEVELOPER, APPLICATIONS",15-1132,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,...,N,Y,BOTH,None,Disclose Business,None,None,None,None,None
4,I-200-19268-394079,Certified,2019-09-25,2019-10-01,None,H-1B,QUALITY ENGINEER LEVEL II,15-1199,"COMPUTER OCCUPATIONS, ALL OTHER",Y,...,N,Y,BOTH,None,Disclose Business,None,None,None,None,LEGAL@THEEGIANTS.COM
5,I-200-19268-495825,Certified,2019-09-25,2019-10-01,None,H-1B,OPERATION RESEARCH ANALYSTS,15-2031,OPERATIONS RESEARCH ANALYSTS,Y,...,N,Y,BOTH,None,Disclose Business,None,None,None,None,None
6,I-200-19268-319248,Certified,2019-09-25,2019-10-01,None,H-1B,SOFTWARE DEVELOPER,15-1132,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,...,N,Y,BOTH,None,Disclose Business,None,None,None,None,None
7,I-200-19268-664806,Certified,2019-09-25,2019-10-01,None,H-1B,.NET LEAD DEVELOPER,15-1132,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,...,N,Y,BOTH,None,Disclose Business,KHAMI,ELIZABETH,S,LAW OFFICES OF RONA M. LUM,LIZ@CORPIMMIGRATION.US
8,I-200-19268-986237,Certified,2019-09-25,2019-10-01,None,H-1B,NETWORK ENGINEER,15-1142,NETWORK AND COMPUTER SYSTEMS ADMINISTRATORS,Y,...,N,Y,BOTH,None,Disclose Business,None,None,None,None,None
9,I-200-19268-150434,Certified,2019-09-25,2019-10-01,None,H-1B,SOFTWARE DEVELOPER (.NET FULL STACK DEVELOPER),15-1132,"SOFTWARE DEVELOPERS, APPLICATIONS",Y,...,N,Y,BOTH,None,Disclose Business,None,None,None,None,None


In [8]:
query = """
SELECT
    CASE_STATUS,
    VISA_CLASS,
    PREVAILING_WAGE,
    WAGE_RATE_OF_PAY_FROM,
    WAGE_UNIT_OF_PAY
FROM lca_disclosure_data
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,CASE_STATUS,VISA_CLASS,PREVAILING_WAGE,WAGE_RATE_OF_PAY_FROM,WAGE_UNIT_OF_PAY
0,Certified,H-1B,95118.0,100000.00,Year
1,Certified,H-1B,39.0,38.57,Hour
2,Certified,H-1B,39.0,43.50,Hour
3,Certified,H-1B,53.0,57.69,Hour
4,Certified,H-1B,65333.0,75000.00,Year
5,Certified,H-1B,72280.0,73000.00,Year
6,Certified,H-1B,73091.0,84000.00,Year
7,Certified,H-1B,73923.0,105000.00,Year
8,Certified,H-1B,74381.0,83000.00,Year
9,Certified,H-1B,74963.0,75000.00,Year


### Se hizo una selección inicial de columnas para desarrollar el proyecto

In [9]:
use_cols = [
    # Target y unidad del target
    "WAGE_RATE_OF_PAY_FROM",
    "WAGE_UNIT_OF_PAY",

    # Salario de referencia regulatorio
    "PREVAILING_WAGE",
    "PW_UNIT_OF_PAY",
    "PW_WAGE_LEVEL",
    "PW_OES_YEAR",
    "PW_OTHER_SOURCE",

    # Variables temporales
    "RECEIVED_DATE",
    "DECISION_DATE",
    "BEGIN_DATE",
    "END_DATE",

    # Datos del empleador y del trabajo
    "EMPLOYER_NAME",
    "EMPLOYER_STATE",
    "WORKSITE_STATE",
    "WORKSITE_CITY",
    "WORKSITE_COUNTY",

    # Visa y ocupación
    "VISA_CLASS",
    "JOB_TITLE",
    "SOC_CODE",
    "SOC_TITLE",
    "NAICS_CODE",

    # Condiciones del empleo
    "FULL_TIME_POSITION",
    "TOTAL_WORKER_POSITIONS",
    "NEW_EMPLOYMENT",
    "CONTINUED_EMPLOYMENT",
    "CHANGE_PREVIOUS_EMPLOYMENT",
    "NEW_CONCURRENT_EMPLOYMENT",
    "CHANGE_EMPLOYER",
    "AMENDED_PETITION",

    # Lugar de trabajo
    "TOTAL_WORKSITE_LOCATIONS",
    "WORKSITE_WORKERS",
    "SECONDARY_ENTITY",

    # Indicadores regulatorios del empleador
    "H_1B_DEPENDENT",
    "WILLFUL_VIOLATOR",
    "SUPPORT_H1B",
    "STATUTORY_BASIS",

    # Estado del caso
    "CASE_STATUS"
]

In [10]:
# Convertir la lista de columnas a texto SQL
cols_sql = ", ".join([f'"{col}"' for col in use_cols])

query = f"""
SELECT {cols_sql}
FROM lca_disclosure_data;
"""

# Creamos df desde SQLite
df = pd.read_sql_query(query, conn)

In [ ]:
# Limpiar variables auxiliares que ya no se necesitan
del cols_sql
del query

# Forzar limpieza de memoria
gc.collect()

In [11]:
df.shape

(3564698, 37)

------------------------
------------------------


## Paso 4: Realiza un análisis descriptivo

In [12]:
df.head()

,WAGE_RATE_OF_PAY_FROM,WAGE_UNIT_OF_PAY,PREVAILING_WAGE,PW_UNIT_OF_PAY,PW_WAGE_LEVEL,PW_OES_YEAR,PW_OTHER_SOURCE,RECEIVED_DATE,DECISION_DATE,BEGIN_DATE,...,CHANGE_EMPLOYER,AMENDED_PETITION,TOTAL_WORKSITE_LOCATIONS,WORKSITE_WORKERS,SECONDARY_ENTITY,H_1B_DEPENDENT,WILLFUL_VIOLATOR,SUPPORT_H1B,STATUTORY_BASIS,CASE_STATUS
0,100000.00,Year,95118.0,Year,IV,2018.0,OES,2019-09-25,2019-10-01,2019-10-07,...,0,0,NaN,NaN,N,N,N,None,None,Certified
1,38.57,Hour,39.0,Hour,II,2019.0,OES,2019-09-25,2019-10-01,2020-01-08,...,0,0,NaN,NaN,Y,Y,N,Y,BOTH,Certified
2,43.50,Hour,39.0,Hour,II,2019.0,OES,2019-09-25,2019-10-01,2019-10-03,...,1,0,NaN,NaN,Y,Y,N,Y,BOTH,Certified
3,57.69,Hour,53.0,Hour,IV,2019.0,OES,2019-09-25,2019-10-01,2019-10-07,...,0,0,NaN,NaN,Y,Y,N,Y,BOTH,Certified
4,75000.00,Year,65333.0,Year,II,2019.0,OES,2019-09-25,2019-10-01,2019-10-09,...,0,0,NaN,NaN,Y,Y,N,Y,BOTH,Certified


In [13]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
WAGE_RATE_OF_PAY_FROM,3564696.0,112164.622733,650897.510530,7.25,80779.0,104749.0,139000.0,1.204781e+09
PREVAILING_WAGE,3562736.0,96229.573737,42041.329473,7.25,74443.0,94390.0,119122.0,8.108500e+05
NAICS_CODE,3564698.0,424819.490078,208648.607037,-271021.00,334413.0,541511.0,541512.0,5.698555e+06
TOTAL_WORKER_POSITIONS,3564698.0,1.762165,5.898817,1.00,1.0,1.0,1.0,1.098000e+03
NEW_EMPLOYMENT,3564698.0,0.635176,3.915777,0.00,0.0,0.0,1.0,1.098000e+03
CONTINUED_EMPLOYMENT,3564698.0,0.371526,2.468180,0.00,0.0,0.0,1.0,1.098000e+03
CHANGE_PREVIOUS_EMPLOYMENT,3564698.0,0.148591,1.311127,0.00,0.0,0.0,0.0,1.098000e+03
NEW_CONCURRENT_EMPLOYMENT,3564698.0,0.010089,0.723727,0.00,0.0,0.0,0.0,1.098000e+03
CHANGE_EMPLOYER,3564698.0,0.304038,1.626940,0.00,0.0,0.0,0.0,1.098000e+03
AMENDED_PETITION,3564698.0,0.299699,1.490564,0.00,0.0,0.0,0.0,1.098000e+03


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3564698 entries, 0 to 3564697
Data columns (total 37 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   WAGE_RATE_OF_PAY_FROM       float64
 1   WAGE_UNIT_OF_PAY            object 
 2   PREVAILING_WAGE             float64
 3   PW_UNIT_OF_PAY              object 
 4   PW_WAGE_LEVEL               object 
 5   PW_OES_YEAR                 object 
 6   PW_OTHER_SOURCE             object 
 7   RECEIVED_DATE               object 
 8   DECISION_DATE               object 
 9   BEGIN_DATE                  object 
 10  END_DATE                    object 
 11  EMPLOYER_NAME               object 
 12  EMPLOYER_STATE              object 
 13  WORKSITE_STATE              object 
 14  WORKSITE_CITY               object 
 15  WORKSITE_COUNTY             object 
 16  VISA_CLASS                  object 
 17  JOB_TITLE                   object 
 18  SOC_CODE                    object 
 19  SOC_TITLE            

### Analizamos columnas con valores nulos

In [15]:
col_nulls = pd.DataFrame(df.isnull().sum()).sort_values(0,ascending=False)
col_nulls

,0
PW_OTHER_SOURCE,3332574
STATUTORY_BASIS,2571646
SUPPORT_H1B,2565205
PW_WAGE_LEVEL,248298
PW_OES_YEAR,230239
WILLFUL_VIOLATOR,91302
H_1B_DEPENDENT,91280
WORKSITE_WORKERS,13335
TOTAL_WORKSITE_LOCATIONS,13333
PREVAILING_WAGE,1962


### Analizamos columnas con alto % de valores nulos

In [16]:
df["PW_OTHER_SOURCE"].value_counts()

,count
PW_OTHER_SOURCE,
Survey,185369
CBA,34805
OES,10917
SURVEY,355
SCA,353
DBA,285
PWD,40


In [17]:
df["STATUTORY_BASIS"].value_counts()


,count
STATUTORY_BASIS,
"$60,000 or higher annual wage",565412
"Both $60,000 or higher in annual wage and Masters Degree or higher in related specialty",204491
WAGE,141356
BOTH,53515
Wage,17410
Both,7226
Masters Degree or higher in related specialty,2475
DEGREE,1093
Degree,74


In [18]:
df["SUPPORT_H1B"].value_counts()


,count
SUPPORT_H1B,
Yes,772384
Y,221127
No,4555
N,1427


In [19]:
df["PW_WAGE_LEVEL"].value_counts()

,count
PW_WAGE_LEVEL,
II,1508404
III,734226
I,540676
IV,533073
V,21


In [20]:
df["PW_OES_YEAR"].value_counts()


,count
PW_OES_YEAR,
7/1/2023 - 6/30/2024,794161
7/1/2020 - 6/30/2021,695132
7/1/2021 - 6/30/2022,624346
7/1/2022 - 6/30/2023,591171
7/1/2019 - 6/30/2020,458999
7/1/2024 - 6/30/2025,94860
10/08/2020 - 6/30/2021,64873
2019.0,4876
2020.0,2502


## Paso 5: Realiza un EDA completo

### Normalizando la columna "PW_OES_YEAR"

In [21]:
def extraer_pw_oes_period(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    years = re.findall(r"\b(20\d{2})\b", value)

    if len(years) >= 2:
        return f"{years[0]}-{years[1]}"

    if len(years) == 1:
        start_year = int(years[0])
        return f"{start_year}-{start_year + 1}"

    return np.nan


df["pw_oes_period_group"] = df["PW_OES_YEAR"].apply(extraer_pw_oes_period)

In [22]:
valid_periods = [
    "2018-2019",
    "2019-2020",
    "2020-2021",
    "2021-2022",
    "2022-2023",
    "2023-2024",
    "2024-2025"
]

df["pw_oes_period_group"] = df["pw_oes_period_group"].where(
    df["pw_oes_period_group"].isin(valid_periods) | df["pw_oes_period_group"].isna(),
    np.nan
)

In [23]:
df["pw_oes_period_group"].value_counts(dropna=False).sort_index()

,count
pw_oes_period_group,
2018-2019,1106
2019-2020,465235
2020-2021,763579
2021-2022,624346
2022-2023,591171
2023-2024,794161
2024-2025,94860
NaN,230240


#### La variable `PW_OES_YEAR` fue transformada en `pw_oes_period_group`, conservando el periodo salarial de referencia en formato `YYYY-YYYY`. Esta decisión facilita la interpretación del análisis, ya que la variable original representa periodos de vigencia salarial y no años calendario aislados.

In [24]:
cols_to_drop = [
    "PW_OTHER_SOURCE",
    "STATUTORY_BASIS",
    "SUPPORT_H1B",
    "PW_OES_YEAR"
]

In [25]:
df = df.drop(columns=cols_to_drop, errors="ignore")

#### Se han eliminado inicialmente las 4 columnas con mayor cantidad de datos nulos y que con previo análisis confirmamos que no entregan información relevante para el modelo.

In [26]:
df.shape

(3564698, 34)

---------------------------------------
---------------------------------------

### Analisis y limpieza de la columna "SOC_TITLE"

In [27]:
df["SOC_TITLE"].value_counts()


,count
SOC_TITLE,
"Software Developers, Applications",633356
Software Developers,527845
Computer Systems Analysts,194637
Computer Systems Engineers/Architects,147930
"Software Developers, Systems Software",120693
...,...
Social And Community Service Managers,1
Water/Wastewater Engineer,1
Validation Engineers,1


---------------------------
---------------------------


## Analisis y Limpieza de la columna "SOC_CODE"

In [28]:
df["SOC_CODE"].value_counts()

,count
SOC_CODE,
15-1132.00,597833
15-1252.00,525445
15-1121.00,120539
15-1133.00,114309
15-1299.08,85680
...,...
49-9061.00,1
43-4031.01,1
31-2011.00,1


In [29]:
top_soc_combinations = (
    df
    .groupby(["SOC_CODE", "SOC_TITLE"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(30)
)

display(top_soc_combinations)

,SOC_CODE,SOC_TITLE,count
662,15-1132.00,"Software Developers, Applications",597648
989,15-1252.00,Software Developers,525286
556,15-1121.00,Computer Systems Analysts,120383
694,15-1133.00,"Software Developers, Systems Software",114253
1089,15-1299.08,Computer Systems Engineers/Architects,85616
60,11-3021.00,Computer and Information Systems Managers,77186
895,15-1211.00,Computer Systems Analysts,64902
837,15-1199.02,Computer Systems Engineers/Architects,59637
1382,17-2141.00,Mechanical Engineers,58961
1131,15-2031.00,Operations Research Analysts,57711


## Se normalizarán los datos de SOC_CODE ya que hay códigos repetidos en diferentes formatos y código contaminado con texto.

In [30]:
# Limpiar SOC_CODE
df["soc_code_clean"] = (
    df["SOC_CODE"]
    .astype("string")
    .str.strip()
    .str.extract(r"(\d{2}-\d{4}(?:\.\d{2})?)")[0]
)
# Eliminar SOC_CODE original
df = df.drop(columns=["SOC_CODE"], errors="ignore")

In [31]:
print("Categorías en soc_code_clean:")
print(df["soc_code_clean"].nunique())

Categorías en soc_code_clean:
1420


In [32]:
# Frecuencia de cada código SOC limpio
soc_code_counts = df["soc_code_clean"].value_counts(dropna=False)

# Distribución de frecuencias
soc_code_freq_distribution = (
    soc_code_counts
    .value_counts()
    .sort_index()
    .rename_axis("cantidad_de_apariciones")
    .reset_index(name="cantidad_de_codigos")
)

display(soc_code_freq_distribution.head(10))

,cantidad_de_apariciones,cantidad_de_codigos
0,1,147
1,2,74
2,3,59
3,4,43
4,5,32
5,6,26
6,7,28
7,8,21
8,9,19
9,10,27


### Distribución de valores de "soc_code_clean"

In [33]:
soc_code_freq_df = soc_code_counts.reset_index()
soc_code_freq_df.columns = ["soc_code_clean", "count"]

fig = px.histogram(
    soc_code_freq_df,
    x="count",
    nbins=100,
    log_y=True,
    title="Distribución de frecuencias de soc_code_clean",
    labels={
        "count": "Cantidad de registros por código SOC",
        "y": "Cantidad de códigos SOC"
    }
)

fig.update_layout(
    xaxis_title="Cantidad de veces que aparece un código SOC",
    yaxis_title="Cantidad de códigos SOC, escala log",
    height=500
)

fig.show()

In [34]:
thresholds = [100, 250, 500, 1000, 1500, 5000, 10000]

rare_summary_soc_code = []

for t in thresholds:
    rare_categories = (soc_code_counts < t).sum()
    rare_rows = soc_code_counts[soc_code_counts < t].sum()

    rare_summary_soc_code.append({
        "umbral_menor_a": t,
        "codigos_agrupados": rare_categories,
        "porcentaje_codigos": round(rare_categories / len(soc_code_counts) * 100, 2),
        "filas_agrupadas": rare_rows,
        "porcentaje_filas": round(rare_rows / len(df) * 100, 2)
    })

rare_summary_soc_code_df = pd.DataFrame(rare_summary_soc_code)

display(rare_summary_soc_code_df)

,umbral_menor_a,codigos_agrupados,porcentaje_codigos,filas_agrupadas,porcentaje_filas
0,100,898,63.19,18129,0.51
1,250,1024,72.06,38282,1.07
2,500,1126,79.24,75202,2.11
3,1000,1197,84.24,127097,3.57
4,1500,1243,87.47,184174,5.17
5,5000,1332,93.74,420022,11.78
6,10000,1368,96.27,679422,19.06


In [35]:
min_count = 1500

valid_soc_codes = df["soc_code_clean"].value_counts()
valid_soc_codes = valid_soc_codes[valid_soc_codes >= min_count].index

df["soc_code_grouped"] = df["soc_code_clean"].where(
    df["soc_code_clean"].isin(valid_soc_codes),
    "OTHER"
)

df = df.drop(columns=["SOC_CODE", "soc_code_clean"], errors="ignore")


In [36]:
print("Categorías finales en soc_code_grouped:")
print(df["soc_code_grouped"].nunique())

Categorías finales en soc_code_grouped:
179


In [37]:
print("Filas agrupadas como OTHER:")
print((df["soc_code_grouped"] == "OTHER").sum())

Filas agrupadas como OTHER:
184174


In [38]:
df["soc_code_grouped"].value_counts()

,count
soc_code_grouped,
15-1132.00,599606
15-1252.00,525445
OTHER,184174
15-1121.00,120539
15-1133.00,114438
...,...
13-2011,1536
15-1141,1523
29-1062.00,1517


### Graficamos la distribución final

In [39]:
soc_code_grouped_counts = df["soc_code_grouped"].value_counts(dropna=False)
soc_code_grouped_freq_df = soc_code_grouped_counts.reset_index()
soc_code_grouped_freq_df.columns = ["soc_code_grouped", "count"]

fig = px.histogram(
    soc_code_grouped_freq_df,
    x="count",
    nbins=100,
    log_y=True,
    title="Distribución de frecuencias de soc_code_grouped",
    labels={
        "count": "Cantidad de registros por código SOC",
        "y": "Cantidad de códigos SOC agrupados"
    }
)

fig.update_layout(
    xaxis_title="Cantidad de veces que aparece un código SOC",
    yaxis_title="Cantidad de códigos SOC agrupados",
    height=500
)

fig.update_yaxes(
    tickmode="array",
    tickvals=[1, 10, 100, 1000],
    ticktext=["1", "10", "100", "1.000"]
)

fig.show()

La variable `SOC_CODE` presentaba alta cardinalidad y formatos inconsistentes, como códigos combinados con texto, generando `soc_code_clean`.

Luego se analizó la frecuencia de cada código limpio. Se observó que muchos códigos aparecían muy pocas veces, por lo que se creó `soc_code_grouped`, manteniendo solo códigos con al menos 500 registros y agrupando el resto como `OTHER`.

Con este umbral se agrupa el 79.24% de los códigos de categorías de trabajos, pero solo el 2.11% de las filas totales, reduciendo dimensionalidad sin afectar significativamente la información del dataset.

------------------------------
------------------------------

## A raiz de las columnas con datos de fechas "RECEIVED_DATE", "DECISION_DATE", "BEGIN_DATE", "END_DATE", las convertiremos en datos útiles numéricos.
- Duración total del trámite.
- Duración total del contrato.

In [40]:
# Convertir fechas a formato datetime
date_cols = ["RECEIVED_DATE", "DECISION_DATE", "BEGIN_DATE", "END_DATE"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

# Crear año de recepción de la solicitud
df["received_year"] = df["RECEIVED_DATE"].dt.year
df["received_year"] = df["received_year"].astype("category")

# 1. Total del trámite en días
df["process_duration_days"] = (
    df["DECISION_DATE"] - df["RECEIVED_DATE"]
).dt.days

# 2. Total del contrato de trabajo en días
df["employment_duration_days"] = (
    df["END_DATE"] - df["BEGIN_DATE"]
).dt.days

# Eliminar columnas originales de fecha, excepto RECEIVED_DATE
df.drop(
    columns=["RECEIVED_DATE","DECISION_DATE", "BEGIN_DATE", "END_DATE"],
    inplace=True,
    errors="ignore"
)

In [41]:
df["process_duration_days"].describe()

,process_duration_days
count,3.564698e+06
mean,2.537339e+01
std,1.086625e+02
min,0.000000e+00
25%,7.000000e+00
50%,7.000000e+00
75%,7.000000e+00
max,1.738000e+03


In [42]:
df["employment_duration_days"].describe()


,employment_duration_days
count,3.564698e+06
mean,1.062578e+03
std,1.295357e+02
min,1.000000e+00
25%,1.094000e+03
50%,1.095000e+03
75%,1.095000e+03
max,1.096000e+03


------------------------------
------------------------------

## Creación de variable objetivo a raiz de la columna WAGE_RATE_OF_PAY_FROM y WAGE_UNIT_OF_PAY

Haremos una conversión de valores donde normalizaremos los valores desde días, semanas, bi-semanas y horas, a que todos los valores queden anuales.

In [43]:
df.head()

,WAGE_RATE_OF_PAY_FROM,WAGE_UNIT_OF_PAY,PREVAILING_WAGE,PW_UNIT_OF_PAY,PW_WAGE_LEVEL,EMPLOYER_NAME,EMPLOYER_STATE,WORKSITE_STATE,WORKSITE_CITY,WORKSITE_COUNTY,...,WORKSITE_WORKERS,SECONDARY_ENTITY,H_1B_DEPENDENT,WILLFUL_VIOLATOR,CASE_STATUS,pw_oes_period_group,soc_code_grouped,received_year,process_duration_days,employment_duration_days
0,100000.00,Year,95118.0,Year,IV,"JO-ANN STORES, INC.",OH,OH,Hudson,Summit,...,NaN,N,N,N,Certified,2018-2019,15-1199,2019,6,1096
1,38.57,Hour,39.0,Hour,II,DENKEN SOLUTIONS INC.,CA,TN,BRENTWOOD,WILLIAMSON,...,NaN,Y,Y,N,Certified,2019-2020,15-1132,2019,6,1095
2,43.50,Hour,39.0,Hour,II,"EPITEC, INC.",MI,MI,Dearborn,Wayne,...,NaN,Y,Y,N,Certified,2019-2020,17-2141,2019,6,1095
3,57.69,Hour,53.0,Hour,IV,"SYSTEMS TECHNOLOGY GROUP, INC.",MI,MI,Taylor,Wayne,...,NaN,Y,Y,N,Certified,2019-2020,15-1132,2019,6,1090
4,75000.00,Year,65333.0,Year,II,E-GIANTS TECHNOLOGIES LLC,IA,OH,BLUE ASH,HAMILTON,...,NaN,Y,Y,N,Certified,2019-2020,15-1199,2019,6,1095


---------------------------
---------------------------

In [44]:
df["WAGE_UNIT_OF_PAY"].value_counts(dropna=False)

,count
WAGE_UNIT_OF_PAY,
Year,3344992
Hour,212172
Month,4519
Week,1739
Bi-Weekly,1274
None,2


In [45]:
# 2. Normalizar la unidad de pago
df["wage_unit_clean"] = (
    df["WAGE_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 3. Diccionario de conversión a salario anual
unit_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080

}
# 4. Crear multiplicador
df["wage_unit_multiplier"] = df["wage_unit_clean"].map(unit_multiplier)

# 5. Convertir salario base a numérico
df["WAGE_UNIT_OF_PAY"] = pd.to_numeric(
    df["WAGE_RATE_OF_PAY_FROM"],
    errors="coerce"
)

# 6. Crear variable objetivo anualizada
df["offered_anual_avg_wage"] = (
    df["WAGE_RATE_OF_PAY_FROM"] * df["wage_unit_multiplier"]
)

df = df.drop(
    columns=["WAGE_RATE_OF_PAY_FROM", "WAGE_UNIT_OF_PAY", "wage_unit_clean", "wage_unit_multiplier"],
    errors="ignore"
)

In [46]:
display(df["offered_anual_avg_wage"].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).apply('{:.2f}'.format))


,offered_anual_avg_wage
count,3564696.00
mean,279716.82
std,6803787.90
min,11.80
1%,46508.80
5%,60000.00
10%,69553.00
25%,85000.00
50%,106950.00
75%,140000.00


### Para reducir los outliers, filtramos dentro del rango de sueldos más razonables.

In [ ]:
# Filtra y muestra las filas en el rango deseado
df_limit_wage = df[df["offered_anual_avg_wage"].between(15000, 300000)].copy()
df_limit_wage = df_limit_wage.reset_index(drop=True)

In [ ]:
df_limit_wage["offered_anual_avg_wage"].describe()


-----------------------------
-----------------------------

## Aplicaremos la misma lógica a la columna "PREVAILING_WAGE"

In [ ]:
# 1. Normalizar la unidad de pago del Prevailing Wage
df_limit_wage["pw_unit_clean"] = (
    df_limit_wage["PW_UNIT_OF_PAY"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 2. Diccionario de conversión a salario anual
unit_multiplier = {
    "YEAR": 1,
    "MONTH": 12,
    "BI-WEEKLY": 26,
    "WEEK": 52,
    "HOUR": 2080
}

# 3. Crear multiplicador (Usamos fillna(1) para que si no hay unidad, asuma que ya es anual)
df_limit_wage["pw_unit_multiplier"] = df_limit_wage["pw_unit_clean"].map(unit_multiplier).fillna(1)

# 4. Convertir salario legal (Prevailing Wage) a numérico de forma segura
df_limit_wage["PREVAILING_WAGE"] = pd.to_numeric(
    df_limit_wage["PREVAILING_WAGE"],
    errors="coerce"
)

# 5. Sobrescribir la variable para que quede anualizada
df_limit_wage["PREVAILING_WAGE"] = (
    df_limit_wage["PREVAILING_WAGE"] * df_limit_wage["pw_unit_multiplier"]
)

# 6. Limpiar la "basura" temporal para no ensuciar el modelo
df_limit_wage = df_limit_wage.drop(
    columns=["pw_unit_clean", "pw_unit_multiplier", "PW_UNIT_OF_PAY"],
    errors="ignore"
)

# 7.
df_limit_wage = df_limit_wage[df_limit_wage["PREVAILING_WAGE"].between(15000, 300000)].copy()
df_limit_wage = df_limit_wage.reset_index(drop=True)

----------------------------
----------------------------

## Analizamos la columna "NEW_EMPLOYMENT"

In [ ]:
df_limit_wage["NEW_EMPLOYMENT"].value_counts()

In [ ]:
df_limit_wage[df_limit_wage["NEW_EMPLOYMENT"]>0]["NEW_EMPLOYMENT"].value_counts()

## En base al análsis previos, tomamos la decisión de trabajar cuando NEW_EMPLOYMENT > 0 y CASE_STATUS == "Certified", para asegurar que la solicitud sea de nuevos empleados y solicitudes certificadas.

In [ ]:
df_limit_wage = df_limit_wage[
    (df_limit_wage["CASE_STATUS"] == "Certified") &
    (df_limit_wage["NEW_EMPLOYMENT"] > 0) &
    (df_limit_wage["TOTAL_WORKER_POSITIONS"] == df_limit_wage["NEW_EMPLOYMENT"])
].copy()

print("Dimensiones del dataset filtrado:")
print(df_limit_wage.shape)


#### Añadimos la siguiente lógica donde "TOTAL_WORKER_POSITIONS" == "NEW_EMPLOYMENT" así garantizamos trabajar exclusivamente con los procesos puramente de nuevos empleados.

In [ ]:
df_limit_wage["NEW_EMPLOYMENT"].value_counts().sort_index()

In [ ]:
print("Validación:")
print("Casos Certificados:", (df_limit_wage["CASE_STATUS"] == "Certified").all())
print("Todos con NEW_EMPLOYMENT > 0:", (df_limit_wage["NEW_EMPLOYMENT"] > 0).all())
print(
    "TOTAL_WORKER_POSITIONS == NEW_EMPLOYMENT:",
    (df_limit_wage["TOTAL_WORKER_POSITIONS"] == df_limit_wage["NEW_EMPLOYMENT"]).all()
)

Se filtró el dataset para trabajar únicamente con solicitudes certificadas correspondientes exclusivamente a nuevos empleados. Esta decisión permite enfocar el proyecto en un escenario empresarial claro: estimar el salario anual mínimo ofrecido en procesos de nueva contratación de talento extranjero.

Para ello, se conservaron solo los registros donde CASE_STATUS es Certified, NEW_EMPLOYMENT es mayor que cero y donde TOTAL_WORKER_POSITIONS coincide con NEW_EMPLOYMENT. Esto asegura que la solicitud no mezcla otros tipos de peticiones, como continuidad laboral(CONTINUED_EMPLOYMENT), cambio de empleador (CHANGE_PREVIOUS_EMPLOYMENT), empleo concurrente (NEW_CONCURRENT_EMPLOYMENT) o peticiones enmendadas (AMENDED_PETITION).

### Validamos columnas con fuerte relación con "NEW_EMPLOYMENT"

In [ ]:
cols_reviews = [
    "CONTINUED_EMPLOYMENT",
    "CHANGE_PREVIOUS_EMPLOYMENT",
    "NEW_CONCURRENT_EMPLOYMENT",
    "CHANGE_EMPLOYER",
    "AMENDED_PETITION"
]

for col in cols_reviews:
    print(f"\n{col}")
    print(df_limit_wage[col].value_counts(dropna=False).head())

Todavía existen algunos registros donde las otras columnas de petición tienen valores distintos de 0.
Filtraremos los datos para que quede enteramente en valores 0 y eliminaremos esas columnas ya que no nos entregan información importante.

In [ ]:
df_filtrado_final = df_limit_wage[
    (df_limit_wage["CONTINUED_EMPLOYMENT"] == 0) &
    (df_limit_wage["CHANGE_PREVIOUS_EMPLOYMENT"] == 0) &
    (df_limit_wage["NEW_CONCURRENT_EMPLOYMENT"] == 0) &
    (df_limit_wage["CHANGE_EMPLOYER"] == 0) &
    (df_limit_wage["AMENDED_PETITION"] == 0)
].copy()


In [ ]:
df_filtrado_final.head()

### Normalizamos variables categóricas binarias

In [ ]:
# Columnas binarias/categóricas
cols_binarias = [
    "FULL_TIME_POSITION",
    "SECONDARY_ENTITY",
    "WILLFUL_VIOLATOR",
    "H_1B_DEPENDENT"
]
for col in cols_binarias:
    print(f"\n{col}")
    print(df_filtrado_final[col].value_counts(dropna=False))

In [ ]:
# Diccionario de normalización
map_binario = {
    "Yes": "Y",
    "No": "N",
    "Y": "Y",
    "N": "N",
    "yes": "Y",
    "no": "N",
    "YES": "Y",
    "NO": "N"
}

# Aplicar normalización a las 4 columnas
for col in cols_binarias:
    df_filtrado_final[col] = (
        df_filtrado_final[col]
        .astype(str)
        .str.strip()
        .map(map_binario)
    )

In [ ]:
for col in cols_binarias:
    print(f"\n{col}")
    print(df_filtrado_final[col].value_counts(dropna=False))

### Normalizamos la columna PW_WAGE_LEVEL

In [ ]:
df_filtrado_final["PW_WAGE_LEVEL"].value_counts(dropna=False)

In [ ]:
# Diccionario de transformación
map_wage_level = {
    "I": "Junior",
    "II": "Intermedio",
    "III": "Avanzado",
    "IV": "Experto"
}


# Aplicar transformación
df_filtrado_final["PW_WAGE_LEVEL"] = (
    df_filtrado_final["PW_WAGE_LEVEL"]
    .astype(str)
    .str.strip()
    .replace(map_wage_level)
)

In [ ]:
df_filtrado_final["PW_WAGE_LEVEL"].value_counts(dropna=False)

In [ ]:
df_filtrado_final.info()

## Eliminamos columnas que no aportan al modelo, con alta cardinalidad y pueden generar leackage

In [ ]:
df_filtrado_final = df_filtrado_final.drop(
    columns=[
        "CASE_STATUS",
        "JOB_TITLE",
        "CONTINUED_EMPLOYMENT",
        "CHANGE_PREVIOUS_EMPLOYMENT",
        "NEW_CONCURRENT_EMPLOYMENT",
        "CHANGE_EMPLOYER",
        "AMENDED_PETITION",
        "EMPLOYER_NAME",
        "WORKSITE_COUNTY",
        "SOC_TITLE",
        "TOTAL_WORKER_POSITIONS"
    ],
    errors="ignore"
)


In [ ]:
col_cat = df_filtrado_final.select_dtypes(include=['object', 'string']).columns
df_filtrado_final[col_cat] = df_filtrado_final[col_cat].astype('category')

In [ ]:
df_filtrado_final["NAICS_CODE"] = df_filtrado_final["NAICS_CODE"].astype("category")

In [ ]:
df_filtrado_final.info()

---------------------------
---------------------------

In [ ]:
df_final = df_filtrado_final.copy()

# 2. Forzar que la variable objetivo sea un número.
df_final['offered_anual_avg_wage'] = pd.to_numeric(df_final['offered_anual_avg_wage'], errors='coerce')

# 3. np.isfinite() detecta y conserva solo números reales (elimina NaN e Inf)
df_final = df_final[np.isfinite(df_final['offered_anual_avg_wage'])]

df_final = df_final.reset_index(drop=True)

In [ ]:

# Save the limited dataset to a new CSV
output_path = '/content/drive/MyDrive/Colab Notebooks/PROYECTO 4GEEKS/df_final.csv'
df_final.to_csv(output_path, index=False)

print(f'The limited dataset has been saved to: {output_path}')


In [ ]:
# Liberar memoria de dataframes anteriores
import gc

for var in ["df", "df_limit_wage", "df_filtrado_final"]:
    if var in globals():
        del globals()[var]

gc.collect()

In [ ]:
df_final.desccc.

# Paso 6: Construye el modelo y optimízalo

Selección y entrenamiento del algoritmo que mejor se ajuste a la naturaleza del problema. Se incluye un proceso de optimización de hiperparámetros para ajustar el modelo y alcanzar el máximo rendimiento posible en sus métricas de evaluación.

In [ ]:
# Separamos las variables independientes y la variable objetivo en "X" y "y".

X = df_final.drop(columns = ['offered_anual_avg_wage', 'PREVAILING_WAGE'])
y = df_final['offered_anual_avg_wage']

In [ ]:
# Usamos train_test_split para separar la data de testeo y de entrenamiento.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model = xgb.XGBRegressor(
    tree_method="hist",
    device="cuda",
    enable_categorical=True,
    random_state=42
)

model.fit(X_train, y_train)

# 3. Predicción
y_pred = model.predict(X_test)

In [ ]:
# Evaluar el desempeño básico
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE (Error Cuadrático Medio): {mse:,.2f}")
print(f"RMSE (Raíz del Error Cuadrático): ${rmse:,.2f}")
print(f"R2 (Coeficiente de Determinación): {r2:.4f}")

### Generamos un dataset de prueba de 100k datos para evaluar el modelo optimizado

In [ ]:
df_100, _ = train_test_split(df_final, train_size=100000, random_state=42, shuffle=True)
df_100 = df_100.reset_index(drop=True)
df_100.info()

In [ ]:
# Separamos las variables independientes y la variable objetivo en "x" y "y".

X = df_100.drop(columns = ['offered_anual_avg_wage',  'PREVAILING_WAGE'])
y = df_100['offered_anual_avg_wage']

In [ ]:
# Usamos train_test_split para separar la data de testeo y de entrenamiento.

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

## Generamos modelo optimizado utilizando Optuna

In [ ]:
def objective(trial):
    param = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "device": "cuda",
        "enable_categorical": True,
        "random_state": 42,

        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),

        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 10.0),

        "reg_alpha": trial.suggest_float("reg_alpha", 1e-6, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),

        "max_cat_to_onehot": trial.suggest_int("max_cat_to_onehot", 1, 32),
    }

    model = xgb.XGBRegressor(
        **param,
        early_stopping_rounds=50
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    preds = model.predict(X_test)
    return np.sqrt(mean_squared_error(y_test, preds))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

print(f"Mejor RMSE: {study.best_value:,.2f}")
print(f"Mejores parámetros: {study.best_params}")

## Evaluamos dataset final con el estudio generado mediante Optuna

In [ ]:
X = df_final.drop(columns = ['offered_anual_avg_wage', 'PREVAILING_WAGE'])
y = df_final['offered_anual_avg_wage']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
final_params = study.best_params.copy()

# 2. Añadimos los parámetros técnicos que no se optimizan (GPU, categorías, etc.)
final_params.update({
    "tree_method": "hist",
    "device": "cuda",
    "enable_categorical": True,
    "random_state": 42,
    "early_stopping_rounds": 50
})

# 3. Instanciamos el modelo usando el desempaquetado de diccionario (**)
final_model = xgb.XGBRegressor(**final_params)

# 4. Entrenamos el modelo
final_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

# 5. Evaluación final
final_preds = final_model.predict(X_test)
rmse_final = np.sqrt(mean_squared_error(y_test, final_preds))
r2_final = r2_score(y_test, final_preds)


print(f"Métricas del Modelo Final:")
print(f"RMSE: {rmse_final}")
print(f"RMSE: {r2_final}")

---------------------------

## Guardamos Modelo

In [ ]:
"""
# Ruta completa donde se guardará el modelo
categorical_cols = X_train.select_dtypes(include=["category"]).columns.tolist()
feature_names = X_train.columns.tolist()

model_bundle = {
    "model": final_model,
    "feature_names": feature_names,
    "categorical_cols": categorical_cols,
    "target": "offered_anual_avg_wage",
    "model_type": "XGBRegressor_final",
    "metrics": {
        "rmse": rmse_final,
        "r2": r2_final
    }
}

output_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/PROYECTO 4GEEKS/optuna_20trials_model.pkl.gz"
)

with gzip.open(output_path, "wb") as file:
    pickle.dump(model_bundle, file)

peso_mb = output_path.stat().st_size / (1024 ** 2)

print(f"✅ Modelo comprimido guardado correctamente en: {output_path}")
print(f"📦 Peso del archivo comprimido: {peso_mb:.2f} MB")
"""

## Paso 7: Despliega el modelo

Fase final donde se crea una interfaz de usuario, habitualmente una aplicación web (usando Flask o Streamlit), y se implementa en una plataforma en la nube como Render o Heroku. Esto permite que el modelo sea accesible para usuarios finales o clientes potenciales.